# Homework #5 Model Deployment


Caitlin Kontaridis
F-1 Dataset

In [0]:
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F
import itertools
from pyspark.sql.window import Window
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import(
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, log_loss, matthews_corrcoef, balanced_accuracy_score, confusion_matrix
)
import warnings
warnings.filterwarnings("ignore")

In [0]:
df_laptimes = spark.read.csv("/Volumes/gr5069/raw/f1_data/lap_times.csv", header=True)
df_drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)
pit_stops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)
races = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)
races = spark.read.csv("/Volumes/gr5069/raw/f1_data/races.csv", header=True)
drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)
results = spark.read.csv("/Volumes/gr5069/raw/f1_data/results.csv", header=True)
w_finish = spark.read.csv("/Volumes/gr5069/raw/f1_data/2022_driver_standings.csv", header=True)
w_finish = Window.partitionBy("raceId").orderBy("positionOrder")
w_pit = Window.partitionBy("raceId").orderBy("avg_duration")


Features

In [0]:
#avg pit stop duration
def parse_duration_to_seconds(col_name):
    raw = F.col(col_name)
    has_colon = F.instr(raw, ":") > 0
    minutes = F.split(raw, ":")[0].cast("float")
    seconds = F.split(raw, ":")[1].cast("float")
    as_minsec = minutes * 60 + seconds
    as_plain = F.when(raw == r"\N", None).otherwise(raw.cast("float"))
    return F.when(has_colon, as_minsec).otherwise(as_plain)

pit_agg = (
    pit_stops
    .withColumn("duration_s", parse_duration_to_seconds("duration"))
    .groupBy("raceId", "driverId")
    .agg(
        F.avg("duration_s").alias("avg_pit_duration"),
        F.count("*").alias("num_pit_stops")
    )
)

In [0]:
# lap time stats /driver/race
lap_agg = (
    df_laptimes
    .withColumn("milliseconds", F.when(F.col("milliseconds") == r"\N", None).otherwise(F.col("milliseconds").cast("float")))
    .groupBy("raceID", "driverId")
    .agg(
        F.avg("milliseconds"). alias("avg_lap_ms"),
        F.stddev("milliseconds").alias("std_lap_ms"),
        F.min("milliseconds").alias("best_lap_ms"),
        F.count("*").alias("laps_completed")
              
    )
)

In [0]:
# clean results and finish rank
def clean_col(col_name, dtype):
    return F.when(F.col(col_name) == r"\N", None).otherwise(F.col(col_name).cast(dtype))

results_clean = (
    results
    .withColumn("grid", clean_col("grid", "int"))
    .withColumn("positionOrder", clean_col("positionOrder", "int"))
    .withColumn("points", clean_col("points", "int"))
    .withColumn("laps", clean_col("laps", "int"))
    .withColumn("fastestLapSpeed", clean_col("fastestLapSpeed", "float"))
    .withColumn("fastestLapRank", clean_col("rank", "float"))
    .withColumn("finish_rank_in_race", F.rank().over(w_finish))
)

In [0]:
#Join
df = (
    results_clean
    .join(pit_agg, on=["raceId", "driverId"], how="left")
    .join(lap_agg, on=["raceId", "driverId"], how="left")
    .join(
        races.select(
            F.col("raceId"),
            F.col("year").cast("int"),
            F.when(F.col("round") == r"\N", None).otherwise(F.col("round").cast("int")).alias("round"),
            F.col("circuitId")
        ),
        on="raceId", how="left"
    )
)

In [0]:
#Define
FEATURE_COLS = [
    "grid",
    "avg_lap_ms",
    "std_lap_ms",
    "best_lap_ms",
    "laps_completed",
    "avg_pit_duration",
    "num_pit_stops",
    "fastestLapSpeed",
    "fastestLapRank",
    "round",
]
TARGET1 = "podium"   #model 1
TARGET2 = "points_finish"  #model 2

pdf = (
    df
    .withColumn(TARGET1, (F.col("positionOrder")<= 3).cast("int"))
    .withColumn(TARGET2, (F.col("positionOrder")<= 10).cast("int"))
    .select("raceId", "driverId", *FEATURE_COLS, TARGET1, TARGET2)
    .dropna(subset=FEATURE_COLS)
    .toPandas()
)

for c in FEATURE_COLS:
    pdf[c] = pd.to_numeric(pdf[c], errors="coerce")

pdf.dropna(subset=FEATURE_COLS, inplace=True)
pdf.reset_index(drop=True, inplace=True)
print(f"Dataset read: {pdf.shape[0]:,} rows")
print(f"Dataset read: {pdf.shape[0]:,} rows, Podium rate: {pdf[TARGET1].mean():.1%}, Points rate: {pdf[TARGET2].mean():.1%}")
X = pdf[FEATURE_COLS]

Create Tables

In [0]:
YOUR_SCHEMA = "cmk2240"
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS gr5069.{YOUR_SCHEMA}.f1_podium_predictions (
              raceID INT,
              driverID INT,
              grid INT,
              avg_lap_ms DOUBLE,
              num_pit_stops INT,
              actual_podium INT,
              predicted_podium INT,
              prediction_probability DOUBLE,
              correct_prediction INT,
              model_name STRING,
              mlflow_run_id STRING
    )
""")
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS gr5069.{YOUR_SCHEMA}.f1_points_predictions (
              raceID INT,
              driverID INT,
              grid INT,
              avg_lap_ms DOUBLE,
              num_pit_stops INT,
              actual_points_finish INT,
              predicted_points_finish INT,
              prediction_probability DOUBLE,
              correct_prediction INT,
              model_name STRING,
              mlflow_run_id STRING
    )       
""")
print(f" Tables created in gr5069.{YOUR_SCHEMA}")

Artifact Helper Functions

In [0]:
def plot_confusion_matrix(y_true, y_pred, title, path):
  cm = confusion_matrix(y_true, y_pred)
  fig, ax = plt.subplots(figsize=(5, 4))
  sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax, xticklabels=["No", "Yes"], yticklabels=["No", "Yes"])
  ax.set_xlabel("Predicted"); ax.set_ylabel("Actual"); ax.set_title(title)
  plt.tight_layout(); fig.savefig(path, dpi=120); plt.close(fig)

def plot_feature_importance(model, feature_names, title, path):
  imp = pd.Series(model.feature_importances_, index=feature_names).sort_values()
  fig, ax = plt.subplots(figsize=(7, 5))
  imp.plot(kind="barh", ax=ax, color="steelblue")
  ax.set_title(title); ax.set_xlabel("Importance")
  plt.tight_layout(); fig.savefig(path, dpi=120); plt.close(fig)

Model 1: Random Forest (Podium)

In [0]:
mlflow.set_experiment("/Users/cmk2240@columbia.edu/F1_Podium_Prediction")

y1 = pdf[TARGET1]
X_tr1, X_te1, y_tr1, y_te1 = train_test_split(
    X, y1, test_size=0.25, random_state=42, stratify=y1
)
test_meta1 = pdf.loc[X_te1.index].reset_index(drop=True)

params_rf = {
    "n_estimators": 200,
    "max_depth": 8,
    "min_samples_split": 10,
    "min_samples_leaf": 4,
    "max_features": "sqrt",
    "class_weight": "balanced",
    "random_state": 42,
}

In [0]:
with mlflow.start_run(run_name="RandomForest_Podium") as run1:
    run_id1 = run1.info.run_id
    rf = RandomForestClassifier(**params_rf)
    rf.fit(X_tr1, y_tr1)
    y_pred1 = rf.predict(X_te1)
    y_proba1 = rf.predict_proba(X_te1)[:, 1]
    mlflow.log_params(params_rf)
    mlflow.log_metrics({"accuracy": round(accuracy_score(y_te1, y_pred1), 4), "f1_score": round(f1_score(y_te1, y_pred1, zero_division=0), 4), "precision": round(precision_score(y_te1, y_pred1, zero_division=0), 4), "recall": round(recall_score(y_te1, y_pred1, zero_division=0), 4), "roc_auc": round(roc_auc_score(y_te1, y_proba1), 4)})
    sig = infer_signature(X_tr1, rf.predict(X_tr1))
    mlflow.sklearn.log_model(rf, "random_forest_podium", signature=sig, registered_model_name="F1_Podium_RF")
    plot_confusion_matrix(y_te1, y_pred1, "Random Forest - Podium", "/tmp/rf_cm.png")
    mlflow.log_artifact("/tmp/rf_cm.png", "plots")
    plot_feature_importance(rf, FEATURE_COLS, "Random Forest - Feature Importances", "/tmp/rf_fi.png")
    mlflow.log_artifact("/tmp/rf_fi.png", "plots")
    print(f"Run ID: {run_id1}")
    print(classification_report(y_te1, y_pred1, target_names=["No Podium", "Podium"]))

Model 1 Predictions


In [0]:
spark.sql(f"DROP TABLE IF EXISTS gr5069.{YOUR_SCHEMA}.f1_podium_predictions")
spark.sql(f"DROP TABLE IF EXISTS gr5069.{YOUR_SCHEMA}.f1_points_predictions")

In [0]:
pred1 = test_meta1[["raceId", "driverId", "grid", "avg_lap_ms", "num_pit_stops", TARGET1]].copy()
pred1.rename(columns={TARGET1: "actual_podium"}, inplace=True)
pred1["predicted_podium"] = y_pred1
pred1["prediction_probability"] = y_proba1
pred1["correct_prediction"] = (y_pred1 == y_te1.values).astype(int)
pred1["model_name"] = "RandomForestClassifier"
pred1["mlflow_run_id"] = run_id1

(spark.createDataFrame(pred1)
    .write
    .mode("overwrite")
    .saveAsTable(f"gr5069.{YOUR_SCHEMA}.f1_podium_predictions"))
print(f" {len(pred1)} rows written to table gr5069.{YOUR_SCHEMA}.f1_podium_prediction")

Model 2 Gradient Boosting Points Finish


In [0]:
mlflow.set_experiment("/Users/cmk2240@columbia.edu/F1_Points_Prediction")

y2 = pdf[TARGET2]
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(
    X, y2, test_size=0.25, random_state=42, stratify=y2
)
test_meta2 = pdf.loc[X_te2.index].reset_index(drop=True)

params_gb = {
    "n_estimators": 150,
    "learning_rate": 0.08,
    "max_depth": 5,
    "subsample": 0.8,
    "min_samples_split": 15,
    "min_samples_leaf": 5,
    "random_state": 42
}
with mlflow.start_run(run_name="GradientBoosting_Points") as run2:
    run_id2 = run2.info.run_id

    gb = GradientBoostingClassifier(**params_gb)
    gb.fit(X_tr2, y_tr2)
    y_pred2 = gb.predict(X_te2)
    y_proba2 = gb.predict_proba(X_te2)[:, 1]

    mlflow.log_params(params_gb)

    cv_f1 = cross_val_score(gb, X_tr2, y_tr2, cv=5, scoring="f1"). mean()
    mlflow.log_metrics({
        "accuracy": round(accuracy_score(y_te2, y_pred2), 4),
        "f1_score": round(f1_score(y_te2, y_pred2, zero_division=0), 4),
        "precision": round(precision_score(y_te2, y_pred2, zero_division=0), 4),
        "recall": round(recall_score(y_te2, y_pred2, zero_division=0), 4),
        "roc_auc": round(roc_auc_score(y_te2, y_proba2), 4),
        "cv_f1": round(cv_f1, 4)
    })
    sig = infer_signature(X_tr2, gb.predict(X_tr2))
    mlflow.sklearn.log_model(gb, "gradient_boosting_points", signature=sig, registered_model_name="F1_Points_GB")
    
    plot_confusion_matrix(y_te2, y_pred2,
                          "Gradient Boosting - Points",
                          "/tmp/gb_cm.png")
    mlflow.log_artifact("/tmp/gb_cm.png", "plots")
    plot_feature_importance(gb, FEATURE_COLS, "Gradient Boosting - Feature Importances", "/tmp/gb_fi.png")
    mlflow.log_artifact("/tmp/gb_fi.png", "plots")
    print(f"Run ID: {run_id2}")
    print(classification_report(y_te2, y_pred2, target_names=["No Points", "Points"]))

Model 2 Predictions to DB

In [0]:
pred2 = test_meta2[["raceId", "driverId", "grid", "avg_lap_ms", "num_pit_stops", TARGET2]].copy()
pred2.rename(columns={TARGET2: "actual_points_finish"}, inplace=True)
pred2["predicted_points_finish"] = y_pred2
pred2["prediction_probability"] = y_proba2
pred2["correct_prediction"] = (y_pred2 == y_te2.values).astype(int)
pred2["model_name"] = "GradientBoostingClassifier"
pred2["mlflow_run_id"] = run_id2

(spark.createDataFrame(pred2)
    .write
    .mode("overwrite")
    .saveAsTable(f"gr5069.{YOUR_SCHEMA}.f1_points_predictions"))
print(f" {len(pred2)} rows written to table gr5069.{YOUR_SCHEMA}.f1_points_prediction")

Validation queries

In [0]:
display(spark.sql(f"""
                  SELECT model_name, mlflow_run_id,
                    COUNT(*) AS total_predictions,
                    ROUND(AVG(correct_prediction) * 100, 2) AS accuracy_pct,
                    ROUND(AVG(prediction_probability), 4) AS avg_confidence
                  FROM gr5069.{YOUR_SCHEMA}.f1_podium_predictions
                  GROUP BY model_name, mlflow_run_id
"""))

display(spark.sql(f"""
                  SELECT model_name, mlflow_run_id,
                  COUNT(*) AS total_predictions,
                   ROUND(AVG(correct_prediction) * 100, 2) AS accuracy_pct,
                   ROUND(AVG(prediction_probability), 4) AS avg_confidence
                  FROM gr5069.{YOUR_SCHEMA}.f1_points_predictions
                  GROUP BY model_name, mlflow_run_id
"""))